<a href="https://colab.research.google.com/github/futuregunmulju/2026-Signals-and-Systems-Speech-Noise-Reduction/blob/experiment%2Fminju/Challenge1_3rd_Model_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== 셀 1: 준비 (v2 + SPP 하이브리드 잡음추정기 + 데이터 + 기준점) =====
import os, sys, json, time, random, importlib
os.environ["OMP_NUM_THREADS"] = "1"
!pip -q install pesq soundfile > /dev/null 2>&1
import numpy as np, pandas as pd, soundfile as sf
from pesq import pesq
from scipy.ndimage import gaussian_filter1d
from scipy.signal import butter, sosfiltfilt, stft
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge"
os.chdir(BASE); sys.path.insert(0, BASE)
import denoise as dn; importlib.reload(dn)         # BASE 안의 v2 (사이렌+하모닉 포함)
assert hasattr(dn, "_harmonic_mult") and hasattr(dn, "_siren_track"), "BASE의 denoise.py가 v2가 아니에요"
N_TRAIN, SEED = 160, 0
STATE = os.path.join(BASE, "search_spp_state.json")

# ---------- Gerkmann–Hendriks 계열 SPP 잡음추정 (앞방향 + 뒤방향) → 기존 추정치와 혼합 ----------
_ORIG_PSD = dn._noise_psd

def _spp_pass(P, N0, xi_db, alpha, bias):
    xi = 10 ** (xi_db / 10); k = xi / (1 + xi)
    sig = N0[:, 0].copy(); pbar = np.full(P.shape[0], 0.5)
    out = np.empty_like(P)
    for t in range(P.shape[1]):
        post = 1.0 / (1.0 + (1 + xi) * np.exp(-(P[:, t] / np.maximum(sig, 1e-14)) * k))   # P(음성 존재 | y)
        pbar = 0.9 * pbar + 0.1 * post
        post = np.where(pbar > 0.99, np.minimum(post, 0.99), post)                        # 정체(stagnation) 방지
        en = (1 - post) * P[:, t] + post * sig                                            # 잡음 파워 MMSE 추정
        sig = np.maximum(alpha * sig + (1 - alpha) * en, 1e-14)
        out[:, t] = sig
    return out * bias

def _hybrid_psd(P, p):
    N0 = _ORIG_PSD(P, p)
    w0 = p.get("spp_w", 0.0)
    if w0 <= 0:
        return N0
    a, xdb, b = p.get("spp_alpha", 0.8), p.get("spp_xi_db", 15.0), p.get("spp_bias", 1.0)
    f = _spp_pass(P, N0, xdb, a, b)
    r = _spp_pass(P[:, ::-1], N0[:, ::-1], xdb, a, b)[:, ::-1]
    Ns = gaussian_filter1d(np.exp(0.5 * (np.log(f) + np.log(r))), p.get("spp_sigma", 15.0), axis=1, mode="nearest")
    # 클립이 비정상(시변)일수록 SPP 쪽 비중을 올리는 게이트
    K = P.shape[0]; e = np.linspace(0, K, 9).astype(int)
    L = 10 * np.log10(np.array([Ns[e[i]:e[i + 1]].mean(axis=0) for i in range(8)]) + 1e-14)
    ns = float(L.std(axis=1).mean())
    gate = min(max((ns - p.get("spp_ns0", 0.0)) / max(p.get("spp_ns_scale", 1.0), 1e-3), 0.0), 1.0)
    w = w0 * gate
    return np.exp((1 - w) * np.log(N0) + w * np.log(np.maximum(Ns, 1e-14)))

dn._noise_psd = _hybrid_psd          # spp_w=0 이면 v2와 완전히 동일

# ---------- 데이터 ----------
def read_pair(row):
    c, _ = sf.read(f"{BASE}/{row.clean_path}", dtype="float64")
    n, _ = sf.read(f"{BASE}/{row.noisy_path}", dtype="float64")
    return c, n

tr_df = pd.read_csv(f"{BASE}/train/samples.csv"); va_df = pd.read_csv(f"{BASE}/validation/samples.csv")
rng = np.random.RandomState(SEED); parts = []
for _, g in tr_df.groupby(["noise_category", "target_snr_db"]):
    k = max(1, int(round(N_TRAIN * len(g) / len(tr_df))))
    parts.append(g.sample(min(k, len(g)), random_state=rng))
tr_sub = pd.concat(parts).reset_index(drop=True)
CLEAN, NOISY, META = [], [], []
for df in (tr_sub, va_df):
    for r in df.itertuples():
        c, n = read_pair(r); CLEAN.append(c); NOISY.append(n); META.append((r.noise_category, r.target_snr_db))
TR = list(range(len(tr_sub))); VA = list(range(len(tr_sub), len(tr_sub) + len(va_df)))
VCAT = np.array([m[0] for m in META[len(tr_sub):]]); VSNR = np.array([m[1] for m in META[len(tr_sub):]])
print("Train 부분집합:", len(tr_sub), "| Validation:", len(va_df))

def _one(args):
    i, params = args
    params = dict(params); mode = params.pop("_oracle", None)
    if mode:                                           # 진단용: 정답 잡음 PSD 주입
        nz = NOISY[i] - CLEAN[i]
        if dn.DEFAULTS["hp_hz"] > 0:
            nz = sosfiltfilt(butter(4, dn.DEFAULTS["hp_hz"], btype="highpass", fs=16000, output="sos"), nz)
        nf, hp = int(dn.DEFAULTS["n_fft"]), int(dn.DEFAULTS["hop"])
        Pn = np.abs(stft(nz, fs=16000, window="hann", nperseg=nf, noverlap=nf - hp)[2]) ** 2
        Nor = np.repeat(Pn.mean(1, keepdims=True), Pn.shape[1], 1) if mode == "mean" else gaussian_filter1d(Pn, 10, axis=1, mode="nearest")
        Nor = np.maximum(Nor, 1e-14)
        dn._noise_psd = lambda P, p, Nor=Nor: Nor
    try:
        y = dn.denoise(NOISY[i], 16000, **params)
    finally:
        dn._noise_psd = _hybrid_psd
    y = np.round(np.clip(y, -1, 1) * 32767) / 32767
    try:
        return pesq(16000, CLEAN[i], y, "wb")
    except Exception:
        return 1.0

import multiprocessing as mp
NW = os.cpu_count() or 2
POOL = mp.get_context("fork").Pool(NW)
def evaluate(params, idx, per_file=False):
    sc = POOL.map(_one, [(i, params) for i in idx], chunksize=2)
    return np.array(sc) if per_file else float(np.mean(sc))

t0 = time.time(); base_tr = evaluate({}, TR); t_eval = time.time() - t0
base_va_arr = evaluate({}, VA, per_file=True); base_va = float(base_va_arr.mean())
print(f"v2 기준점  Train부분집합 {base_tr:.4f} | Validation {base_va:.4f}  (기대: 1.8395)")
print(f"Train 1회 평가 {t_eval:.1f}초 → 1시간에 약 {int(3600/t_eval)}회 후보 평가")

Mounted at /content/drive
Train 부분집합: 171 | Validation: 164
v2 기준점  Train부분집합 1.7936 | Validation 1.8395  (기대: 1.8395)
Train 1회 평가 50.9초 → 1시간에 약 70회 후보 평가


In [ ]:
# ===== 셀 2: 진단 — 우리 잡음에 시변 추적이 필요한가? (약 5~10분) =====
# (a) 잡음 비정상도: 잡음 신호만 보고, 8개 대역 파워가 0.1초 평활 후 시간에 따라 얼마나(dB) 흔들리는지
def nonstat_db(nz):
    nz = sosfiltfilt(butter(4, 100, btype="highpass", fs=16000, output="sos"), nz)
    P = np.abs(stft(nz, fs=16000, window="hann", nperseg=640, noverlap=480)[2]) ** 2
    e = np.logspace(np.log10(100), np.log10(8000), 9); f = np.linspace(0, 8000, P.shape[0])
    L = []
    for i in range(8):
        m = (f >= e[i]) & (f < e[i + 1]); L.append(10 * np.log10(P[m].mean(0) + 1e-14))
    L = gaussian_filter1d(np.array(L), 5, axis=1, mode="nearest")[:, 5:-5]
    return float(L.std(axis=1).mean())

ns_rows = []
for r in tr_df.itertuples():
    c, n = read_pair(r); ns_rows.append((r.noise_category, nonstat_db(n - c)))
ns_df = pd.DataFrame(ns_rows, columns=["cat", "ns"]).groupby("cat").ns.agg(["mean", "count"])

# (b) 정답 잡음 PSD를 줬을 때 PESQ 이득: 클립 평균(수준 오차) / 시변(시간 추적)
cur = base_va_arr
om = evaluate({"_oracle": "mean"}, VA, per_file=True)
ot = evaluate({"_oracle": "tv"}, VA, per_file=True)
d = pd.DataFrame({"cat": VCAT, "cur": cur, "수준갭": om - cur, "시변갭": ot - om})
tab = d.groupby("cat").agg(n=("cur", "size"), 현재=("cur", "mean"), 수준갭=("수준갭", "mean"), 시변갭=("시변갭", "mean"))
tab["비정상도dB(Train)"] = ns_df["mean"].reindex(tab.index)
tab = tab.sort_values("n", ascending=False)
print(f"전체: 현재 {cur.mean():.3f} | 정답PSD(클립평균) {om.mean():.3f} | 정답PSD(시변) {ot.mean():.3f}")
display(tab[tab.n >= 4].round(3))
print("읽는 법: 비정상도 큰 카테고리에서 '시변갭'이 크면 SPP 계열이 유망. '수준갭'만 크면 추적이 아니라 수준 추정 정확도 문제.")
print("주의: 정답PSD는 실제 잡음 실현값이라 상한이며 블라인드 추정기가 다 가져올 수는 없음.")

전체: 현재 1.840 | 정답PSD(클립평균) 1.933 | 정답PSD(시변) 2.088


,n,현재,수준갭,시변갭,비정상도dB(Train)
cat,,,,,
machine,34,1.824,0.098,0.176,3.215
wind,26,2.009,0.037,0.084,1.495
alarm_siren,23,1.737,0.191,0.158,4.948
aircraft_cockpit,14,1.668,0.056,0.071,0.746
broadband_noise,10,1.455,-0.011,0.038,0.547
traffic_vehicle,9,1.879,0.181,0.407,3.935
destroyer_engine,5,1.833,-0.030,0.064,1.049
water,5,1.312,0.128,0.223,6.114
military_vehicle,5,2.110,-0.022,0.107,1.093


읽는 법: 비정상도 큰 카테고리에서 '시변갭'이 크면 SPP 계열이 유망. '수준갭'만 크면 추적이 아니라 수준 추정 정확도 문제.
주의: 정답PSD는 실제 잡음 실현값이라 상한이며 블라인드 추정기가 다 가져올 수는 없음.


In [ ]:
# ===== 셀 3: SPP 하이브리드 탐색 (초기 격자 → 지역 탐색) =====
TIME_H = 3.0
SPACE = {   # (하한, 상한)
    "spp_w": (0.0, 1.0), "spp_sigma": (2.0, 40.0), "spp_xi_db": (8.0, 20.0), "spp_alpha": (0.6, 0.95),
    "spp_bias": (0.8, 1.6), "spp_ns0": (0.0, 6.0), "spp_ns_scale": (0.5, 8.0),
    "noise_scale": (0.9, 1.8), "noise_sigma": (20.0, 90.0), "floor_pause_db": (-40.0, -20.0),
}
SPP0 = dict(spp_w=0.0, spp_sigma=15.0, spp_xi_db=15.0, spp_alpha=0.8, spp_bias=1.0, spp_ns0=0.0, spp_ns_scale=1.0)
v2 = {k: dn.DEFAULTS[k] for k in ("noise_scale", "noise_sigma", "floor_pause_db")}
base_params = {**SPP0, **v2}

def perturb(cur, frac, r):
    new = dict(cur)
    for k in r.sample(list(SPACE), r.choice([1, 2, 3])):
        lo, hi = SPACE[k]
        new[k] = float(min(max(new[k] + r.gauss(0, (0.02 + 0.10 * (1 - frac)) * (hi - lo)), lo), hi))
    return new

if os.path.exists(STATE):
    S = json.load(open(STATE)); print("이어서:", S["iters"], "회, Train 최고", round(S["best_tr"], 4))
else:
    # 1) 초기 격자: SPP가 조금이라도 도움이 되는지 빠르게 확인
    print("초기 격자 (spp_w × spp_sigma, 나머지 기본):")
    best, best_s = None, -1
    for w in (0.3, 0.6, 0.9):
        for sg in (5.0, 15.0, 30.0):
            c = dict(base_params, spp_w=w, spp_sigma=sg); s = evaluate(c, TR)
            print(f"  w={w} sigma={sg:>4}: Train {s:.4f} (v2 {base_tr:.4f}, {s-base_tr:+.4f})")
            if s > best_s: best, best_s = c, s
    S = dict(best=best, best_tr=best_s, iters=0, elapsed=0.0,
             ckpts=[dict(params=dict(base_params), tr=base_tr, va=base_va, it=0)])
json.dump(S, open(STATE, "w"))
r = random.Random(4242 + S["iters"]); last_ck = max(base_tr, max(c["tr"] for c in S["ckpts"]))
t_start = time.time() - S["elapsed"]
try:
    while time.time() - t_start < TIME_H * 3600:
        frac = min(1.0, (time.time() - t_start) / (TIME_H * 3600))
        cand = perturb(S["best"], frac, r); s = evaluate(cand, TR); S["iters"] += 1
        if s > S["best_tr"] + 1e-5:
            S["best"], S["best_tr"] = cand, s; json.dump(S, open(STATE, "w"))
            if s - last_ck >= 0.002:
                v = evaluate(cand, VA); S["ckpts"].append(dict(params=dict(cand), tr=s, va=v, it=S["iters"])); last_ck = s
                print(f"  ★ it={S['iters']}: Train {s:.4f} | Validation {v:.4f} (v2 {base_va:.4f}, {v-base_va:+.4f})")
        if S["iters"] % 10 == 0:
            S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE, "w"))
            print(f"it={S['iters']:4d} | {S['elapsed']/3600:.2f}h | Train 최고 {S['best_tr']:.4f} (v2 {base_tr:.4f})")
except KeyboardInterrupt:
    print("중단됨 (저장됨)")
S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE, "w")); print("종료. 총", S["iters"], "회")

초기 격자 (spp_w × spp_sigma, 나머지 기본):
  w=0.3 sigma= 5.0: Train 1.7961 (v2 1.7936, +0.0025)
  w=0.3 sigma=15.0: Train 1.7929 (v2 1.7936, -0.0007)
  w=0.3 sigma=30.0: Train 1.7920 (v2 1.7936, -0.0016)
  w=0.6 sigma= 5.0: Train 1.7907 (v2 1.7936, -0.0028)
  w=0.6 sigma=15.0: Train 1.7893 (v2 1.7936, -0.0042)
  w=0.6 sigma=30.0: Train 1.7884 (v2 1.7936, -0.0052)
  w=0.9 sigma= 5.0: Train 1.7774 (v2 1.7936, -0.0162)
  w=0.9 sigma=15.0: Train 1.7770 (v2 1.7936, -0.0166)
  w=0.9 sigma=30.0: Train 1.7781 (v2 1.7936, -0.0154)
  ★ it=7: Train 1.8049 | Validation 1.8518 (v2 1.8395, +0.0123)
it=  10 | 0.33h | Train 최고 1.8059 (v2 1.7936)
  ★ it=11: Train 1.8079 | Validation 1.8563 (v2 1.8395, +0.0168)
it=  20 | 0.65h | Train 최고 1.8095 (v2 1.7936)
  ★ it=25: Train 1.8099 | Validation 1.8587 (v2 1.8395, +0.0191)
it=  30 | 0.97h | Train 최고 1.8100 (v2 1.7936)
it=  40 | 1.26h | Train 최고 1.8103 (v2 1.7936)
it=  50 | 1.55h | Train 최고 1.8113 (v2 1.7936)
it=  60 | 1.85h | Train 최고 1.8116 (v2 1.7936)
it=  70 |

In [ ]:
# ===== 셀 4: 결과 확인 + 카테고리별로 어디서 올랐는지 =====
S = json.load(open(STATE))
S["ckpts"].append(dict(params=dict(S["best"]), tr=S["best_tr"], va=evaluate(S["best"], VA), it=S["iters"]))
for c in S["ckpts"]:
    print(f"it={c['it']:4d} | Train {c['tr']:.4f} | Validation {c['va']:.4f} | spp_w {c['params']['spp_w']:.2f}")
bk = max(S["ckpts"], key=lambda c: c["va"]); gain = bk["va"] - base_va
print(f"\nv2 {base_va:.4f} → SPP 하이브리드 {bk['va']:.4f} ({gain:+.4f})")
arr = evaluate(bk["params"], VA, per_file=True)
t = pd.DataFrame({"cat": VCAT, "d": arr - base_va_arr}).groupby("cat").d.agg(["size", "mean"])
display(t[t["size"] >= 4].sort_values("mean").round(3))
print("SNR별 변화:", pd.Series(arr - base_va_arr).groupby(VSNR).mean().round(3).to_dict())
json.dump({**{k: dn.DEFAULTS[k] for k in dn.DEFAULTS}, **bk["params"]}, open(os.path.join(BASE, "best_params_spp.json"), "w"), indent=2)
print("판정:", "채택 후보 (+0.005 이상)" if gain >= 0.005 else "v2 유지")
print(bk["params"])

it=   0 | Train 1.7936 | Validation 1.8395 | spp_w 0.00
it=   7 | Train 1.8049 | Validation 1.8518 | spp_w 0.30
it=  11 | Train 1.8079 | Validation 1.8563 | spp_w 0.30
it=  25 | Train 1.8099 | Validation 1.8587 | spp_w 0.40
it=  75 | Train 1.8120 | Validation 1.8607 | spp_w 0.31
it=  99 | Train 1.8123 | Validation 1.8609 | spp_w 0.31

v2 1.8395 → SPP 하이브리드 1.8609 (+0.0214)


,size,mean
cat,,
phone_dialing,4,-0.011
military_vehicle,5,-0.009
destroyer_operations,5,-0.004
destroyer_engine,5,0.009
alarm_siren,23,0.017
machine,34,0.018
water,5,0.019
traffic_vehicle,9,0.022
hf_channel,4,0.025


SNR별 변화: {0.0: 0.01, 5.0: 0.021, 10.0: 0.012, 15.0: 0.034}
판정: 채택 후보 (+0.005 이상)
{'spp_w': 0.3068935374498033, 'spp_sigma': 2.0, 'spp_xi_db': 14.62247855028253, 'spp_alpha': 0.8336048442600156, 'spp_bias': 1.6, 'spp_ns0': 0.10145702026474249, 'spp_ns_scale': 0.6711397164760309, 'noise_scale': 1.589896186320649, 'noise_sigma': 48.26717981051946, 'floor_pause_db': -28.091237542032918}


### 확인실험

In [2]:
# 어디서 오른 건지 분해 (약 10분) -> 셀1을 실행해야지 사용 가능함.
S = json.load(open(STATE)); b = dict(S["best"])
v2 = {k: dn.DEFAULTS[k] for k in ("noise_scale", "noise_sigma", "floor_pause_db")}
cases = {
  "전체 (SPP + 재튜닝)": b,
  "SPP 끄고 재튜닝값만 (spp_w=0)": dict(b, spp_w=0.0),
  "SPP만 (noise_scale/noise_sigma/floor_pause는 v2 값)": dict(b, **v2),
}
for name, p in cases.items():
    print(f"{name}: Train {evaluate(p, TR):.4f} | Validation {evaluate(p, VA):.4f}")

전체 (SPP + 재튜닝): Train 1.8123 | Validation 1.8609
SPP 끄고 재튜닝값만 (spp_w=0): Train 1.8077 | Validation 1.8567
SPP만 (noise_scale/noise_sigma/floor_pause는 v2 값): Train 1.8063 | Validation 1.8527


# SPP 잡음 추정 & 파라미터 재튜닝

In [3]:
# ===== 셀 A: 통합 — SPP 하이브리드를 denoise.py에 정식 통합 → BASE/v3/ 에 저장 (기존 파일은 안 건드림, 셀 1 불필요) =====
import os, json
BASE = "/content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge"

s = open(f"{BASE}/denoise.py").read()
assert "_harmonic_mult" in s and "_siren_track" in s, "BASE의 denoise.py가 v2가 아니에요"
assert "_spp_pass" not in s, "이미 통합된 파일이에요"
assert s.count("def _noise_psd(P, p):") == 1

NEW = '''def _spp_pass(P, N0, xi_db, alpha, bias):
    """Gerkmann-Hendriks 계열 SPP 기반 잡음 파워 추적(한 방향)."""
    xi = 10 ** (xi_db / 10)
    k = xi / (1 + xi)
    sig = N0[:, 0].copy()
    pbar = np.full(P.shape[0], 0.5)
    out = np.empty_like(P)
    for t in range(P.shape[1]):
        post = 1.0 / (1.0 + (1 + xi) * np.exp(-(P[:, t] / np.maximum(sig, 1e-14)) * k))  # P(음성 존재|y)
        pbar = 0.9 * pbar + 0.1 * post
        post = np.where(pbar > 0.99, np.minimum(post, 0.99), post)                       # 정체 방지
        en = (1 - post) * P[:, t] + post * sig
        sig = np.maximum(alpha * sig + (1 - alpha) * en, 1e-14)
        out[:, t] = sig
    return out * bias


def _noise_psd(P, p):
    """기존 최소통계 추정(N0)과 양방향 SPP 추정을 로그 영역에서 혼합."""
    N0 = _noise_psd_base(P, p)
    w0 = p["spp_w"]
    if w0 <= 0:
        return N0
    a, xdb, b = p["spp_alpha"], p["spp_xi_db"], p["spp_bias"]
    f = _spp_pass(P, N0, xdb, a, b)
    r = _spp_pass(P[:, ::-1], N0[:, ::-1], xdb, a, b)[:, ::-1]
    Ns = gaussian_filter1d(np.exp(0.5 * (np.log(f) + np.log(r))), p["spp_sigma"], axis=1, mode="nearest")
    K = P.shape[0]
    e = np.linspace(0, K, 9).astype(int)
    L = 10 * np.log10(np.array([Ns[e[i]:e[i + 1]].mean(axis=0) for i in range(8)]) + 1e-14)
    ns = float(L.std(axis=1).mean())            # 입력에서 직접 잰 시간 변동 폭(dB)
    gate = min(max((ns - p["spp_ns0"]) / max(p["spp_ns_scale"], 1e-3), 0.0), 1.0)
    w = w0 * gate
    return np.exp((1 - w) * np.log(N0) + w * np.log(np.maximum(Ns, 1e-14)))


'''
s = s.replace("def _noise_psd(P, p):", "def _noise_psd_base(P, p):", 1)
a = "def _tonal_suppress("
assert s.count(a) == 1
s = s.replace(a, NEW + a, 1)
a = "    h_fmax=2500.0,\n)"
assert s.count(a) == 1
s = s.replace(a, "    h_fmax=2500.0,\n    spp_w=0.0, spp_sigma=15.0, spp_xi_db=15.0, spp_alpha=0.8,\n"
                 "    spp_bias=1.0, spp_ns0=0.0, spp_ns_scale=1.0,\n)", 1)

out = f"{BASE}/v3"; os.makedirs(out, exist_ok=True)
open(f"{out}/denoise.py", "w").write(s)
params = json.load(open(f"{BASE}/best_params_spp.json"))
assert params["spp_w"] > 0 and "siren_w" in params and "h_w" in params
json.dump(params, open(f"{out}/best_params.json", "w"), indent=2, ensure_ascii=False)
open(f"{out}/requirements.txt", "w").write("numpy\nscipy\nsoundfile\n")
print("생성 완료:", out)
print({k: round(v, 4) for k, v in params.items() if k.startswith("spp_") or k in ("noise_scale", "noise_sigma", "floor_pause_db")})

생성 완료: /content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge/v3
{'noise_sigma': 48.2672, 'noise_scale': 1.5899, 'floor_pause_db': -28.0912, 'spp_w': 0.3069, 'spp_sigma': 2.0, 'spp_xi_db': 14.6225, 'spp_alpha': 0.8336, 'spp_bias': 1.6, 'spp_ns0': 0.1015, 'spp_ns_scale': 0.6711}


In [4]:
# ===== 셀 B: 검증 — v3 폴더를 CLI로 돌려 Validation PESQ 확인 (기대 ≈ 1.8609, 셀 1 불필요) =====
# pesq 에러가 나면 맨 위에 !pip -q install pesq soundfile 추가
import os, sys, shutil, subprocess
import numpy as np, pandas as pd, soundfile as sf
from pesq import pesq
BASE = "/content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge"
V3 = f"{BASE}/v3"
va = pd.read_csv(f"{BASE}/validation/samples.csv")
for d in ("/content/v3_in", "/content/v3_out"):
    shutil.rmtree(d, ignore_errors=True)
os.makedirs("/content/v3_in")
for r in va.itertuples():
    shutil.copy(f"{BASE}/{r.noisy_path}", f"/content/v3_in/{r.segment_id}.wav")
subprocess.run([sys.executable, f"{V3}/denoise.py", "--input_dir", "/content/v3_in", "--output_dir", "/content/v3_out"], check=True)
sc = []
for r in va.itertuples():
    c, _ = sf.read(f"{BASE}/{r.clean_path}", dtype="float64")
    y, _ = sf.read(f"/content/v3_out/{r.segment_id}.wav", dtype="float64")
    sc.append(pesq(16000, c, y, "wb"))
va["pesq"] = sc
print(f"CLI Validation PESQ-WB: {np.mean(sc):.4f}  (기대 ≈ 1.8609)")
print(va.groupby("target_snr_db").pesq.mean().round(3).to_dict())

CLI Validation PESQ-WB: 1.8609  (기대 ≈ 1.8609)
{0.0: 1.264, 5.0: 1.593, 10.0: 1.876, 15.0: 2.188}


In [5]:
# ===== 셀 C: 2차 탐색 — 범위 확장, 1차 최고값에서 시작, 약 1.25시간 (셀 1 필요) =====
# 유의미한 차이 없어서 스탑
TIME_H = 1.25
STATE2 = os.path.join(BASE, "search_spp2_state.json")
REF_VA = 1.8609            # 1차 결과(= v3). 이보다 +0.003 이상 올라야 2차 결과를 채택
S1 = json.load(open(STATE))

SPACE2 = {   # 1차에서 상한/하한에 붙었던 spp_bias, spp_sigma, noise_scale 범위를 넓히고, 이득 관련 2개 추가
    "spp_w": (0.0, 1.0), "spp_sigma": (1.0, 40.0), "spp_xi_db": (8.0, 20.0), "spp_alpha": (0.6, 0.95),
    "spp_bias": (0.8, 2.6), "spp_ns0": (0.0, 6.0), "spp_ns_scale": (0.3, 8.0),
    "noise_scale": (0.9, 2.4), "noise_sigma": (20.0, 90.0), "floor_pause_db": (-40.0, -20.0),
    "alpha_dd": (0.90, 0.985), "gain_floor_db": (-30.0, -18.0),
}
start = {k: float(S1["best"][k] if k in S1["best"] else dn.DEFAULTS[k]) for k in SPACE2}
TRH = TR[::2]              # 속도: 탐색 중에는 Train 절반만 사용 (1회 약 50초)

def perturb2(cur, frac, r):
    new = dict(cur)
    for k in r.sample(list(SPACE2), r.choice([1, 2, 3])):
        lo, hi = SPACE2[k]
        new[k] = float(min(max(new[k] + r.gauss(0, (0.02 + 0.10 * (1 - frac)) * (hi - lo)), lo), hi))
    return new

if os.path.exists(STATE2):
    S = json.load(open(STATE2)); print("이어서:", S["iters"], "회, Train(절반) 최고", round(S["best_tr"], 4))
else:
    b0 = evaluate(start, TRH)
    S = dict(best=start, best_tr=b0, iters=0, elapsed=0.0, ckpts=[dict(params=dict(start), tr=b0, va=REF_VA, it=0)])
    print(f"출발점(1차 최고) Train(절반) {b0:.4f}")
json.dump(S, open(STATE2, "w"))
r = random.Random(9090 + S["iters"]); last_ck = max(c["tr"] for c in S["ckpts"])
t_start = time.time() - S["elapsed"]
try:
    while time.time() - t_start < TIME_H * 3600:
        frac = min(1.0, (time.time() - t_start) / (TIME_H * 3600))
        cand = perturb2(S["best"], frac, r); s = evaluate(cand, TRH); S["iters"] += 1
        if s > S["best_tr"] + 1e-5:
            S["best"], S["best_tr"] = cand, s; json.dump(S, open(STATE2, "w"))
            if s - last_ck >= 0.003:
                v = evaluate(cand, VA); S["ckpts"].append(dict(params=dict(cand), tr=s, va=v, it=S["iters"])); last_ck = s
                print(f"  ★ it={S['iters']}: Train(절반) {s:.4f} | Validation {v:.4f} (1차 {REF_VA:.4f}, {v-REF_VA:+.4f})")
        if S["iters"] % 10 == 0:
            S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE2, "w"))
            print(f"it={S['iters']:4d} | {S['elapsed']/3600:.2f}h | Train(절반) 최고 {S['best_tr']:.4f}")
except KeyboardInterrupt:
    print("중단됨 (저장됨)")
S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE2, "w")); print("탐색 종료. 총", S["iters"], "회")

# ---- 결과 정리 ----
arr_last = evaluate(S["best"], VA, per_file=True)
S["ckpts"].append(dict(params=dict(S["best"]), tr=S["best_tr"], va=float(arr_last.mean()), it=S["iters"]))
for c in S["ckpts"]:
    print(f"it={c['it']:4d} | Train(절반) {c['tr']:.4f} | Validation {c['va']:.4f}")
bk2 = max(S["ckpts"], key=lambda c: c["va"]); gain2 = bk2["va"] - REF_VA
print(f"\n1차(v3) {REF_VA:.4f} → 2차 최고 {bk2['va']:.4f} ({gain2:+.4f})")
arr2 = evaluate(bk2["params"], VA, per_file=True)
print("SNR별:", pd.Series(arr2).groupby(VSNR).mean().round(3).to_dict())
json.dump({**{k: dn.DEFAULTS[k] for k in dn.DEFAULTS}, **bk2["params"]}, open(os.path.join(BASE, "best_params_spp2.json"), "w"), indent=2)
print("판정:", "2차 채택 후보 (+0.003 이상)" if gain2 >= 0.003 else "개선폭 미달 → 1차(v3) 유지")
print({k: round(v, 4) for k, v in bk2["params"].items()})

출발점(1차 최고) Train(절반) 1.8739
it=  10 | 0.07h | Train(절반) 최고 1.8739
it=  20 | 0.15h | Train(절반) 최고 1.8750
it=  30 | 0.23h | Train(절반) 최고 1.8751
it=  40 | 0.30h | Train(절반) 최고 1.8751
it=  50 | 0.37h | Train(절반) 최고 1.8752
it=  60 | 0.45h | Train(절반) 최고 1.8752
it=  70 | 0.52h | Train(절반) 최고 1.8753
it=  80 | 0.60h | Train(절반) 최고 1.8754
it=  90 | 0.67h | Train(절반) 최고 1.8755
중단됨 (저장됨)
탐색 종료. 총 94 회


Process ForkPoolWorker-3:


KeyboardInterrupt: 

Traceback (most recent call last):


In [ ]:
# ===== 셀 D: 분해 확인 (약 10분) — 이득이 SPP 덕인지 재조정 덕인지 가림 / 셀 1이 실행된 상태여야 함 =====
S = json.load(open(STATE)); b = dict(S["best"])
v2 = {k: dn.DEFAULTS[k] for k in ("noise_scale", "noise_sigma", "floor_pause_db")}
cases = {
  "전체 (SPP + 재튜닝)": b,
  "SPP 끄고 재튜닝값만 (spp_w=0)": dict(b, spp_w=0.0),
  "SPP만 (noise_scale/noise_sigma/floor_pause는 v2 값)": dict(b, **v2),
}
for name, p in cases.items():
    print(f"{name}: Train {evaluate(p, TR):.4f} | Validation {evaluate(p, VA):.4f}")

# E. 입력 SNR 적응형 파라미터 실험 -> 성능이 더 나아지지 않음. 기각

In [6]:
# ===== 셀 E1: 입력만 보고 잰 SNR 추정이 실제 SNR과 얼마나 맞는지 (약 2분, 셀 1 필요) =====
S_ = json.load(open(STATE)); p_best = {**dn.DEFAULTS, **S_["best"]}

def blind_snr(x, p):
    xh = sosfiltfilt(butter(4, p["hp_hz"], btype="highpass", fs=16000, output="sos"), x) if p["hp_hz"] > 0 else x
    if p["tonal_db"] > 0:
        xh = dn._tonal_suppress(xh, 16000, p["tonal_db"], p["tonal_max_cut_db"], int(p["tonal_max_peaks"]))
    nf, hp = int(p["n_fft"]), int(p["hop"])
    P = np.abs(stft(xh, fs=16000, window="hann", nperseg=nf, noverlap=nf - hp)[2]) ** 2 + 1e-14
    N = dn._noise_psd(P, dict(p))
    return 10 * np.log10(max(P.sum() - N.sum(), 0.02 * P.sum()) / N.sum())

est = np.array([blind_snr(NOISY[i], p_best) for i in VA])
d_ = pd.DataFrame({"실제SNR": VSNR, "추정SNR": est})
print(d_.groupby("실제SNR").추정SNR.agg(["mean", "std", "count"]).round(2))
print("상관계수:", round(float(np.corrcoef(d_["실제SNR"], d_["추정SNR"])[0, 1]), 3))
print("읽는 법: 실제 SNR이 오를 때 추정SNR 평균이 뚜렷이 오르고(상관 0.7 이상) 표준편차가 간격(5dB)보다 작으면 입력 SNR 적응형 파라미터가 의미 있음.")

        mean   std  count
실제SNR                    
0.0    -4.61  6.87     15
5.0     3.17  5.65     38
10.0    8.43  4.30     55
15.0   12.58  4.35     56
상관계수: 0.72
읽는 법: 실제 SNR이 오를 때 추정SNR 평균이 뚜렷이 오르고(상관 0.7 이상) 표준편차가 간격(5dB)보다 작으면 입력 SNR 적응형 파라미터가 의미 있음.


In [7]:
# ===== 셀 E2: 입력 SNR 적응형 파라미터 탐색 (약 1시간, 셀 1 필요, 셀 C가 끝난/멈춘 뒤 실행) =====
TIME_H = 1.0
STATE3 = os.path.join(BASE, "search_snr_state.json")
REF_VA = 1.8609            # v3. 이보다 +0.003 이상 올라야 채택 후보
base1 = json.load(open(STATE))["best"]          # 1차 탐색 최고값(= v3)

# --- 입력 SNR 적응형 잡음추정 래퍼: 파라미터를 입력에서 잰 SNR에 따라 기울기(slope)만큼 이동 ---
if "_HYB0" not in globals():
    _HYB0 = _hybrid_psd                         # 지금까지의 하이브리드 (재실행해도 중복 감싸지 않음)

def _adaptive_psd(P, p):
    N = _HYB0(P, p)
    a_s, a_f, a_p, a_d = p.get("ad_scale", 0.0), p.get("ad_floor", 0.0), p.get("ad_pause", 0.0), p.get("ad_dd", 0.0)
    if a_s == 0.0 and a_f == 0.0 and a_p == 0.0 and a_d == 0.0:
        return N
    snr = 10 * np.log10(max(P.sum() - N.sum(), 0.02 * P.sum()) / N.sum())      # 입력만 보고 잰 SNR(dB)
    z = float(np.clip((snr - p.get("ad_ref", 8.0)) / 10.0, -1.0, 1.5))
    p["gain_floor_db"] = float(np.clip(p["gain_floor_db"] + a_f * z, -40.0, -10.0))
    if p.get("floor_pause_db") is not None:
        p["floor_pause_db"] = float(np.clip(p["floor_pause_db"] + a_p * z, -45.0, -15.0))
    p["alpha_dd"] = float(np.clip(p["alpha_dd"] + a_d * z, 0.85, 0.99))
    return N * np.exp(a_s * z)

_hybrid_psd = _adaptive_psd                     # _one()의 finally가 이 이름으로 복구하므로 같이 교체
dn._noise_psd = _adaptive_psd
POOL.terminate(); POOL = mp.get_context("fork").Pool(NW)    # 새 워커가 새 함수를 상속

SPACE3 = {
    "ad_scale": (-0.6, 0.6), "ad_floor": (-6.0, 6.0), "ad_pause": (-6.0, 6.0), "ad_dd": (-0.04, 0.04), "ad_ref": (0.0, 15.0),
    "noise_scale": (0.9, 2.4), "gain_floor_db": (-30.0, -18.0), "floor_pause_db": (-40.0, -20.0), "alpha_dd": (0.90, 0.985),
}
start = {k: float(base1[k] if k in base1 else dn.DEFAULTS[k]) for k in SPACE3 if not k.startswith("ad_")}
start.update(ad_scale=0.0, ad_floor=0.0, ad_pause=0.0, ad_dd=0.0, ad_ref=8.0)
fixed = {k: base1[k] for k in base1 if k not in SPACE3}          # spp_* 등 나머지는 v3 값 고정
TRH = TR[::2]

def perturb3(cur, frac, r):
    new = dict(cur)
    for k in r.sample(list(SPACE3), r.choice([1, 2, 3])):
        lo, hi = SPACE3[k]
        new[k] = float(min(max(new[k] + r.gauss(0, (0.02 + 0.10 * (1 - frac)) * (hi - lo)), lo), hi))
    return new

ev = lambda params, idx, **kw: evaluate({**fixed, **params}, idx, **kw)
if os.path.exists(STATE3):
    S = json.load(open(STATE3)); print("이어서:", S["iters"], "회, Train(절반) 최고", round(S["best_tr"], 4))
else:
    b0 = ev(start, TRH)
    print(f"출발점 Train(절반) {b0:.4f}  (셀 C 출발점 1.8739와 같아야 정상: 기울기 0이면 v3와 동일)")
    S = dict(best=start, best_tr=b0, iters=0, elapsed=0.0, ckpts=[dict(params=dict(start), tr=b0, va=REF_VA, it=0)])
json.dump(S, open(STATE3, "w"))
r = random.Random(555 + S["iters"]); last_ck = max(c["tr"] for c in S["ckpts"])
t_start = time.time() - S["elapsed"]
try:
    while time.time() - t_start < TIME_H * 3600:
        frac = min(1.0, (time.time() - t_start) / (TIME_H * 3600))
        cand = perturb3(S["best"], frac, r); s = ev(cand, TRH); S["iters"] += 1
        if s > S["best_tr"] + 1e-5:
            S["best"], S["best_tr"] = cand, s; json.dump(S, open(STATE3, "w"))
            if s - last_ck >= 0.003:
                v = ev(cand, VA); S["ckpts"].append(dict(params=dict(cand), tr=s, va=v, it=S["iters"])); last_ck = s
                print(f"  ★ it={S['iters']}: Train(절반) {s:.4f} | Validation {v:.4f} (v3 {REF_VA:.4f}, {v-REF_VA:+.4f})")
        if S["iters"] % 10 == 0:
            S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE3, "w"))
            print(f"it={S['iters']:4d} | {S['elapsed']/3600:.2f}h | Train(절반) 최고 {S['best_tr']:.4f}")
except KeyboardInterrupt:
    print("중단됨 (저장됨)")
S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE3, "w")); print("탐색 종료. 총", S["iters"], "회")

# ---- 결과 정리 ----
arr_last = ev(S["best"], VA, per_file=True)
S["ckpts"].append(dict(params=dict(S["best"]), tr=S["best_tr"], va=float(arr_last.mean()), it=S["iters"]))
for c in S["ckpts"]:
    print(f"it={c['it']:4d} | Train(절반) {c['tr']:.4f} | Validation {c['va']:.4f}")
bk3 = max(S["ckpts"], key=lambda c: c["va"]); gain3 = bk3["va"] - REF_VA
print(f"\nv3 {REF_VA:.4f} → 적응형 최고 {bk3['va']:.4f} ({gain3:+.4f})")
arr3 = ev(bk3["params"], VA, per_file=True)
print("SNR별:", pd.Series(arr3).groupby(VSNR).mean().round(3).to_dict())
json.dump({**{k: dn.DEFAULTS[k] for k in dn.DEFAULTS}, **fixed, **bk3["params"]}, open(os.path.join(BASE, "best_params_snr.json"), "w"), indent=2)
print("판정:", "채택 후보 (+0.003 이상)" if gain3 >= 0.003 else "개선폭 미달 → v3 유지")
print({k: round(v, 4) for k, v in bk3["params"].items()})

출발점 Train(절반) 1.8739  (셀 C 출발점 1.8739와 같아야 정상: 기울기 0이면 v3와 동일)
it=  10 | 0.07h | Train(절반) 최고 1.8741
it=  20 | 0.15h | Train(절반) 최고 1.8741
it=  30 | 0.22h | Train(절반) 최고 1.8742
it=  40 | 0.29h | Train(절반) 최고 1.8742
it=  50 | 0.37h | Train(절반) 최고 1.8744
it=  60 | 0.44h | Train(절반) 최고 1.8750
it=  70 | 0.51h | Train(절반) 최고 1.8758
it=  80 | 0.58h | Train(절반) 최고 1.8759
it=  90 | 0.66h | Train(절반) 최고 1.8760
it= 100 | 0.73h | Train(절반) 최고 1.8760
it= 110 | 0.81h | Train(절반) 최고 1.8760
it= 120 | 0.88h | Train(절반) 최고 1.8760
it= 130 | 0.95h | Train(절반) 최고 1.8760
탐색 종료. 총 137 회
it=   0 | Train(절반) 1.8739 | Validation 1.8609
it= 137 | Train(절반) 1.8761 | Validation 1.8615

v3 1.8609 → 적응형 최고 1.8615 (+0.0006)
SNR별: {0.0: 1.26, 5.0: 1.593, 10.0: 1.875, 15.0: 2.191}
판정: 개선폭 미달 → v3 유지
{'noise_scale': 1.6415, 'gain_floor_db': -24.2043, 'floor_pause_db': -28.7515, 'alpha_dd': 0.9503, 'ad_scale': 0.1742, 'ad_floor': 0.4165, 'ad_pause': 0.2898, 'ad_dd': -0.0007, 'ad_ref': 5.0848}


# F. 파이프라인 모델 하나씩 꺼보고 성능 비교

In [8]:
POOL.close()

In [1]:
# ===== 셀 X0: 준비 — v3에 구조 옵션 3개(OM-LSA식 이득 수정 / 주파수별 이득 하한 / 양방향 DD)를 추가한 실험판을 만들고 기준점 확인 =====
# 새 노트북에서 첫 셀로 실행하세요. BASE/v3/ 에 검증된 v3 파일(denoise.py, best_params.json)이 있어야 해요.
import os, sys, json, time, random, shutil, importlib.util
os.environ["OMP_NUM_THREADS"] = "1"
!pip -q install pesq soundfile > /dev/null 2>&1
import numpy as np, pandas as pd, soundfile as sf
from pesq import pesq
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge"
os.chdir(BASE)
EXP = f"{BASE}/exp"; os.makedirs(EXP, exist_ok=True)
STATE = f"{EXP}/search_x_state.json"
N_TRAIN, SEED = 160, 0

# ---------- v3 denoise.py에 옵션 3개를 텍스트 패치 (모두 끄면 v3와 출력이 완전히 같음) ----------
s = open(f"{BASE}/v3/denoise.py").read()
assert "_spp_pass" in s and "om_w" not in s, "BASE/v3/denoise.py 상태가 예상과 달라요"

a = "    spp_bias=1.0, spp_ns0=0.0, spp_ns_scale=1.0,\n)"
assert s.count(a) == 1
s = s.replace(a, "    spp_bias=1.0, spp_ns0=0.0, spp_ns_scale=1.0,\n    om_w=0.0, om_q=0.4, fl_tilt_db=0.0, bd_w=0.0,\n)", 1)

old = s[s.index("    G = np.empty_like(P)\n    prev = np.zeros(K)\n    for t in range(T):"): s.index("    if p[\"tsnr\"]:")]
assert "prev = g ** 2 * gamma" in old and old.count("for t in range(T)") == 1
new = '''    TV = 10 ** (p["fl_tilt_db"] * (fr / (sr / 2.0) - 0.5) / 20.0)      # 주파수별 이득 하한 기울기(0이면 전부 1.0)

    def _xi_t(t, prev, gamma):
        xi = np.maximum(p["alpha_dd"] * prev + (1 - p["alpha_dd"]) * np.maximum(gamma - 1, 0), xi_min)
        if XC is not None:
            w = p["ceps_w"]
            xi = np.exp((1 - w) * np.log(xi) + w * np.log(XC[:, t]))
        if HM is not None:
            xi = np.maximum(xi * HM[:, t], xi_min)
        return xi

    XB = None
    if p["bd_w"] > 0:                                   # 시간 역방향 decision-directed 패스 (미래 프레임 정보)
        XB = np.empty_like(P)
        prevb = np.zeros(K)
        for t in range(T - 1, -1, -1):
            gamma = np.minimum(P[:, t] / N[:, t], 1000.0)
            xi = _xi_t(t, prevb, gamma)
            XB[:, t] = xi
            gb = np.clip(_lsa(xi, gamma), floor_t[t] * TV, 1.0)
            prevb = gb ** 2 * gamma

    G = np.empty_like(P)
    prev = np.zeros(K)
    for t in range(T):
        gamma = np.minimum(P[:, t] / N[:, t], 1000.0)
        xi = _xi_t(t, prev, gamma)
        if XB is not None:
            xi = np.exp((1 - p["bd_w"]) * np.log(xi) + p["bd_w"] * np.log(XB[:, t]))
        gl = _lsa(xi, gamma)
        ft = floor_t[t] * TV
        g = np.clip(gl, ft, 1.0)
        prev = g ** 2 * gamma                           # DD 재귀는 OM 수정 전 이득을 사용
        if p["om_w"] > 0:                               # OM-LSA식: 음성 존재 확률 pr로 G_LSA^pr * G_floor^(1-pr)
            v = gamma * xi / (1 + xi)
            q = min(max(p["om_q"], 1e-3), 0.999)
            pr = 1.0 / (1.0 + (q / (1 - q)) * (1 + xi) * np.exp(-np.minimum(v, 50.0)))
            g_om = np.clip(np.exp(pr * np.log(np.maximum(gl, 1e-6)) + (1 - pr) * np.log(ft)), ft, 1.0)
            g = np.exp((1 - p["om_w"]) * np.log(g) + p["om_w"] * np.log(g_om))
        G[:, t] = g

'''
s = s.replace(old, new, 1)
open(f"{EXP}/denoise_x.py", "w").write(s)
shutil.copy(f"{BASE}/v3/best_params.json", f"{EXP}/best_params.json")
spec = importlib.util.spec_from_file_location("dnx", f"{EXP}/denoise_x.py"); dnx = importlib.util.module_from_spec(spec); spec.loader.exec_module(dnx)
assert dnx.DEFAULTS["om_w"] == 0.0 and dnx.DEFAULTS["spp_w"] > 0

# ---------- 데이터 ----------
def read_pair(row):
    c, _ = sf.read(f"{BASE}/{row.clean_path}", dtype="float64")
    n, _ = sf.read(f"{BASE}/{row.noisy_path}", dtype="float64")
    return c, n

tr_df = pd.read_csv(f"{BASE}/train/samples.csv"); va_df = pd.read_csv(f"{BASE}/validation/samples.csv")
rng = np.random.RandomState(SEED); parts = []
for _, g in tr_df.groupby(["noise_category", "target_snr_db"]):
    k = max(1, int(round(N_TRAIN * len(g) / len(tr_df))))
    parts.append(g.sample(min(k, len(g)), random_state=rng))
tr_sub = pd.concat(parts).reset_index(drop=True)
CLEAN, NOISY, META = [], [], []
for df in (tr_sub, va_df):
    for r in df.itertuples():
        c, n = read_pair(r); CLEAN.append(c); NOISY.append(n); META.append((r.noise_category, r.target_snr_db))
TR = list(range(len(tr_sub))); VA = list(range(len(tr_sub), len(tr_sub) + len(va_df)))
TRH = TR[::2]
VCAT = np.array([m[0] for m in META[len(tr_sub):]]); VSNR = np.array([m[1] for m in META[len(tr_sub):]])

def _one(args):
    i, params = args
    y = dnx.denoise(NOISY[i], 16000, **params)
    y = np.round(np.clip(y, -1, 1) * 32767) / 32767
    try:
        return pesq(16000, CLEAN[i], y, "wb")
    except Exception:
        return 1.0

import multiprocessing as mp
NW = os.cpu_count() or 2
POOL = mp.get_context("fork").Pool(NW)
def evaluate(params, idx, per_file=False):
    sc = POOL.map(_one, [(i, params) for i in idx], chunksize=2)
    return np.array(sc) if per_file else float(np.mean(sc))

t0 = time.time(); b_h = evaluate({}, TRH); t_eval = time.time() - t0
base_va_arr = evaluate({}, VA, per_file=True); base_va = float(base_va_arr.mean())
print(f"기준점(옵션 전부 끔) Train(절반) {b_h:.4f} | Validation {base_va:.4f}   (기대: 1.8739 / 1.8609, v3와 같아야 정상)")
print(f"Train(절반) 1회 평가 {t_eval:.1f}초 → 1시간에 약 {int(3600/t_eval)}회")

Mounted at /content/drive
기준점(옵션 전부 끔) Train(절반) 1.8739 | Validation 1.8609   (기대: 1.8739 / 1.8609, v3와 같아야 정상)
Train(절반) 1회 평가 28.6초 → 1시간에 약 125회


In [10]:
# ===== 셀 Y1: 구성요소 재검증 — v3의 각 단계를 하나씩 끄거나 바꿔 보고 실제로 도움이 되는지 확인 (약 13분, 셀 X0 필요) =====
# 노트북 B(E2 노트북)에서 X0를 먼저 실행한 뒤 이 셀을 실행하세요. 점수 변화 Δ가 음수면 그 단계가 도움이 되고 있다는 뜻이에요.
abl = [
    ("순음 억제 끔            tonal_db=0",        dict(tonal_db=0.0)),
    ("켑스트럼 SNR 끔         ceps_w=0",          dict(ceps_w=0.0)),
    ("하모닉 comb 끔          h_w=0",             dict(h_w=0.0)),
    ("사이렌 PSD 끔           siren_w=0",         dict(siren_w=0.0)),
    ("이득 평활 끔            gain_smooth=False", dict(gain_smooth=False)),
    ("leak 끔                 leak=0",            dict(leak=0.0)),
    ("일시정지 하한 끔        floor_pause=gain_floor", dict(floor_pause_db=-24.0)),
    ("HPF 끔                  hp_hz=0",           dict(hp_hz=0.0)),
    ("HPF 60 Hz               hp_hz=60",          dict(hp_hz=60.0)),
    ("SPP 끔 (확인용)         spp_w=0",           dict(spp_w=0.0)),
    ("TSNR 켬                 tsnr=True",         dict(tsnr=True)),
    ("잡음 반복 2회           n_iter=2",          dict(n_iter=2)),
    ("창 512/128",                                 dict(n_fft=512, hop=128)),
    ("창 800/200",                                 dict(n_fft=800, hop=200)),
]
res = {}
for name, prm in abl:
    t0 = time.time(); sc = evaluate(prm, TRH); res[name] = (sc, prm)
    print(f"{name:<44} Train(절반) {sc:.4f} ({sc-b_h:+.4f})  [{time.time()-t0:.0f}초]")

good = [(n, v) for n, v in res.items() if v[0] - b_h >= 0.001]
print(f"\n기준 {b_h:.4f}. +0.001 이상 오른 항목:", [n.split('  ')[0].strip() for n, _ in good] or "없음")
if good:
    comb = {}
    for n, (sc, prm) in good: comb.update(prm)
    c_tr = evaluate(comb, TRH); print(f"오른 항목 전부 합침 → Train(절반) {c_tr:.4f} ({c_tr-b_h:+.4f})  설정: {comb}")
    c_va = evaluate(comb, VA); print(f"Validation {c_va:.4f} (v3 {base_va:.4f}, {c_va-base_va:+.4f})")
    json.dump(dict(params=comb, tr=c_tr, va=c_va), open(f"{EXP}/ablation_best.json", "w"), indent=2)
    print("판정:", "채택 후보 (+0.003 이상)" if c_va - base_va >= 0.003 else "개선폭 미달")
else:
    print("끄거나 바꿔서 오른 단계가 없음 → 현재 구성요소는 모두 제 몫을 하고 있음")

순음 억제 끔            tonal_db=0                Train(절반) 1.8730 (-0.0009)  [36초]
켑스트럼 SNR 끔         ceps_w=0                  Train(절반) 1.8419 (-0.0320)  [31초]
하모닉 comb 끔          h_w=0                    Train(절반) 1.8733 (-0.0006)  [28초]
사이렌 PSD 끔           siren_w=0                Train(절반) 1.8706 (-0.0033)  [24초]
이득 평활 끔            gain_smooth=False         Train(절반) 1.8774 (+0.0035)  [28초]
leak 끔                 leak=0                Train(절반) 1.8750 (+0.0011)  [28초]
일시정지 하한 끔        floor_pause=gain_floor      Train(절반) 1.8731 (-0.0008)  [28초]
HPF 끔                  hp_hz=0               Train(절반) 1.8502 (-0.0237)  [27초]
HPF 60 Hz               hp_hz=60             Train(절반) 1.8365 (-0.0374)  [28초]
SPP 끔 (확인용)         spp_w=0                  Train(절반) 1.8681 (-0.0058)  [26초]
TSNR 켬                 tsnr=True             Train(절반) 1.7545 (-0.1194)  [29초]
잡음 반복 2회           n_iter=2                  Train(절반) 1.8728 (-0.0011)  [31초]
창 512/128                                    Train(절

# Z. 성능 오른 조합으로 다시 실험

In [11]:
# ===== 셀 Z1: 창 크기/hop 격자 — Y1에서 오른 두 가지(이득 평활 끔, leak 끔)를 켠 채로 창을 더 키워 봄 (약 5분, 셀 X0 필요) =====
B4 = dict(gain_smooth=False, leak=0.0)
wins = [(640, 160), (800, 200), (800, 160), (960, 240), (960, 160), (1024, 256), (1024, 160), (1280, 320)]
zres = []
for nf, hp in wins:
    prm = dict(B4, n_fft=nf, hop=hp)
    t0 = time.time(); sc = evaluate(prm, TRH); zres.append((sc, prm))
    print(f"n_fft={nf:4d} hop={hp:3d}  Train(절반) {sc:.4f} ({sc-b_h:+.4f})  [{time.time()-t0:.0f}초]")
zb = max(zres, key=lambda t: t[0])
print(f"\n최고: n_fft={zb[1]['n_fft']} hop={zb[1]['hop']}  Train(절반) {zb[0]:.4f} ({zb[0]-b_h:+.4f})")
json.dump(dict(params=zb[1], tr=zb[0], grid=[[s, p] for s, p in zres]), open(f"{EXP}/z1_best.json", "w"), indent=1)
v = evaluate(zb[1], VA); print(f"Validation {v:.4f} (v3 {base_va:.4f}, {v-base_va:+.4f}; Y1 합친 조합 1.9133)")

n_fft= 640 hop=160  Train(절반) 1.8755 (+0.0016)  [47초]
n_fft= 800 hop=200  Train(절반) 1.9002 (+0.0263)  [45초]
n_fft= 800 hop=160  Train(절반) 1.8917 (+0.0178)  [32초]
n_fft= 960 hop=240  Train(절반) 1.9074 (+0.0335)  [27초]
n_fft= 960 hop=160  Train(절반) 1.8994 (+0.0255)  [33초]
n_fft=1024 hop=256  Train(절반) 1.9043 (+0.0304)  [28초]
n_fft=1024 hop=160  Train(절반) 1.9026 (+0.0286)  [35초]
n_fft=1280 hop=320  Train(절반) 1.9023 (+0.0284)  [26초]

최고: n_fft=960 hop=240  Train(절반) 1.9074 (+0.0335)
Validation 1.9206 (v3 1.8609, +0.0597; Y1 합친 조합 1.9133)


In [2]:
# ===== 셀 Z2: 새 창에서 파라미터 재탐색 (약 1시간, 셀 X0, Z1 필요) =====
# 노트북 A와 B에서 각각 실행할 수 있어요. 노트북마다 TAG와 SEED만 다르게 두세요 (B: "b",1 / A: "a",2). 같은 TAG면 상태 파일이 덮어써져요.
TAG, SEED_Z, TIME_H = "b", 1, 1.0
Z1 = json.load(open(f"{EXP}/z1_best.json")); FIX = Z1["params"]       # gain_smooth, leak, n_fft, hop
STATE_Z = f"{EXP}/search_z_{TAG}_state.json"
REF_VA = Z1.get("va", None) or evaluate(FIX, VA)
SPACE = {
    "noise_scale": (0.9, 2.6), "noise_sigma": (15.0, 90.0), "min_bias": (1.4, 3.2), "delta": (2.0, 4.5), "min_win": (60.0, 220.0),
    "alpha_dd": (0.90, 0.985), "gain_floor_db": (-32.0, -16.0), "floor_pause_db": (-42.0, -20.0),
    "ceps_w": (0.0, 0.5), "h_w": (0.1, 0.7), "spp_w": (0.0, 0.6), "tonal_db": (10.0, 30.0),
}
base_p = {**FIX}
start = {k: float(dnx.DEFAULTS[k]) for k in SPACE}
def full(c): return {**base_p, **c}
def perturb(cur, frac, r):
    new = dict(cur)
    for k in r.sample(list(SPACE), r.choice([1, 2, 3])):
        lo, hi = SPACE[k]
        new[k] = float(min(max(new[k] + r.gauss(0, (0.02 + 0.10 * (1 - frac)) * (hi - lo)), lo), hi))
    return new
if os.path.exists(STATE_Z):
    S = json.load(open(STATE_Z)); print("이어서:", S["iters"], "회, Train(절반) 최고", round(S["best_tr"], 4))
else:
    b0 = evaluate(full(start), TRH); print(f"출발점 Train(절반) {b0:.4f} (Z1 최고와 같아야 정상: {Z1['tr']:.4f})")
    S = dict(best=start, best_tr=b0, iters=0, elapsed=0.0, ckpts=[dict(params=full(start), tr=b0, va=REF_VA, it=0)])
json.dump(S, open(STATE_Z, "w"))
r = random.Random(1357 * SEED_Z + S["iters"]); last_ck = max(c["tr"] for c in S["ckpts"])
t_start = time.time() - S["elapsed"]
try:
    while time.time() - t_start < TIME_H * 3600:
        frac = min(1.0, (time.time() - t_start) / (TIME_H * 3600))
        cand = perturb(S["best"], frac, r); s = evaluate(full(cand), TRH); S["iters"] += 1
        if s > S["best_tr"] + 1e-5:
            S["best"], S["best_tr"] = cand, s; json.dump(S, open(STATE_Z, "w"))
            if s - last_ck >= 0.003:
                v = evaluate(full(cand), VA); S["ckpts"].append(dict(params=full(cand), tr=s, va=v, it=S["iters"])); last_ck = s
                print(f"  ★ it={S['iters']}: Train(절반) {s:.4f} | Validation {v:.4f} (시작 {REF_VA:.4f}, {v-REF_VA:+.4f})")
        if S["iters"] % 10 == 0:
            S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE_Z, "w"))
            print(f"it={S['iters']:4d} | {S['elapsed']/3600:.2f}h | Train(절반) 최고 {S['best_tr']:.4f}")
except KeyboardInterrupt:
    print("중단됨 (저장됨)")
S["elapsed"] = time.time() - t_start; json.dump(S, open(STATE_Z, "w")); print("탐색 종료. 총", S["iters"], "회")

이어서: 107 회, Train(절반) 최고 1.9548
  ★ it=109: Train(절반) 1.9550 | Validation 1.9620 (시작 1.9206, +0.0414)
it= 110 | 0.92h | Train(절반) 최고 1.9554
  ★ it=118: Train(절반) 1.9582 | Validation 1.9626 (시작 1.9206, +0.0420)
탐색 종료. 총 119 회


In [3]:
# ===== 셀 Z3: Z2 결과 확인과 판정 (Z2가 끝난 뒤, ■ 정지는 한 번만) =====
S = json.load(open(STATE_Z))
arr = evaluate(S["best"] if False else {**FIX, **S["best"]}, VA, per_file=True)
S["ckpts"].append(dict(params={**FIX, **S["best"]}, tr=S["best_tr"], va=float(arr.mean()), it=S["iters"]))
for c in S["ckpts"]:
    print(f"it={c['it']:4d} | Train(절반) {c['tr']:.4f} | Validation {c['va']:.4f}")
bk = max(S["ckpts"], key=lambda c: c["va"]); gain = bk["va"] - REF_VA
print(f"\n시작(Z1 최고) {REF_VA:.4f} → 최고 {bk['va']:.4f} ({gain:+.4f}) | v3 대비 {bk['va']-base_va:+.4f}")
a2 = evaluate(bk["params"], VA, per_file=True)
print("SNR별:", pd.Series(a2).groupby(VSNR).mean().round(3).to_dict())
d = pd.DataFrame({"cat": VCAT, "d": a2 - base_va_arr}).groupby("cat").d.agg(["size", "mean"])
print("v3 대비 카테고리별 변화(n>=4):"); print(d[d["size"] >= 4].sort_values("mean").round(3))
json.dump({**{k: dnx.DEFAULTS[k] for k in dnx.DEFAULTS}, **bk["params"]}, open(f"{EXP}/best_params_z_{TAG}.json", "w"), indent=2)
print("판정:", "채택 후보 (+0.003 이상)" if gain >= 0.003 else "개선폭 미달 → Z1 최고 설정 유지")
print({k: round(v, 4) if isinstance(v, float) else v for k, v in bk["params"].items()})

it=   0 | Train(절반) 1.9074 | Validation 1.9206
it=  13 | Train(절반) 1.9122 | Validation 1.9233
it=  22 | Train(절반) 1.9156 | Validation 1.9296
it=  23 | Train(절반) 1.9188 | Validation 1.9332
it=  26 | Train(절반) 1.9364 | Validation 1.9529
it=  48 | Train(절반) 1.9395 | Validation 1.9524
it=  56 | Train(절반) 1.9446 | Validation 1.9579
it=  81 | Train(절반) 1.9476 | Validation 1.9601
it=  98 | Train(절반) 1.9520 | Validation 1.9615
it= 109 | Train(절반) 1.9550 | Validation 1.9620
it= 118 | Train(절반) 1.9582 | Validation 1.9626
it= 119 | Train(절반) 1.9582 | Validation 1.9626

시작(Z1 최고) 1.9206 → 최고 1.9626 (+0.0420) | v3 대비 +0.1017
SNR별: {0.0: 1.366, 5.0: 1.704, 10.0: 1.965, 15.0: 2.296}
v3 대비 카테고리별 변화(n>=4):
                      size   mean
cat                              
traffic_vehicle          9  0.003
hf_channel               4  0.038
phone_dialing            4  0.054
factory                  5  0.072
alarm_siren             23  0.078
water                    5  0.102
destroyer_engine         5  0